Train meta learner bằng tập train, early stop bằng valid sau đó test trên tập test.

In [1]:
import pandas as pd
data_train = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\train_df_prepared.parquet")
data_valid = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\valid_df_prepared.parquet")
data_test = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1-exper2\test_df_prepared.parquet")

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
import pandas as pd
import numpy as np
import gc

# mô hình GatEmbedder
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# SE Block: từ đầu ra là các channel từ CNN, kênh nào chiếm tín hiệu rõ ràng thì trọng số là 1, kênh nào bị nhiễu or sai lệch thì trọng số = 0
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # từ các channel của CNN, tính average pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        
        # tính điểm dựa trên 1 mạng neural channel => channel/8 => channel và đi qua hàm sigmoid để tạo thang điểm 0-1 cho từng kênh
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    # b,c: batch size, số kênh
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # tính toán điểm trung bình của các channel theo chiều thời gian
        y = self.fc(y).view(b, c, 1)    # tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # nhân điểm số này với các channel để khuếch đại kênh chuẩn và bóp nghẹt kênh bị nhiễm Drift

# residual block: CNN tích hợp SE Block và shortcut connection 
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding) # đi qua lớp convoluation đầu tiên
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels) # group norm: chuẩn hóa độc lập theo từng nhóm kênh
        self.relu = nn.ReLU() # hàm kích hoạt ReLU
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding) # đi qua lớp convolution thứ hai
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels) # chuẩn hóa sau lớp convolution thứ hai
        self.dropout = nn.Dropout1d(0.2)
        
        # SE Block: chấm điểm đầu ra cho các channel của CNN
        self.se = SEBlock1D(out_channels)
        
        # shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels: # nếu số kênh đầu vào khác số kênh đầu ra thì shortcut phải dùng convolution để điều chỉnh số kênh
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        out = self.se(out)
        out += residual  
        out = self.relu(out)
        return out


# model CNN-BiLSTM hoàn chỉnh
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # 2 khối Residual Block
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # cho BiLSTM nhận đầu vào là 128 channel, tạo ra output có chiều dài hiden_size*2
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # chuẩn hóa layer norm theo từng time-step
        self.layer_norm = nn.LayerNorm(hidden_size * 2)

        # tính attention score cho output của BiLSTM và tạo vector ngữ cảnh (context vector)
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x ban đầu: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) # chuyển thành (Batch, Features, SeqLen) để phù hợp với đầu vào của CNN
        
        # đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        # chuyển lại thành (batch, seq_len, features) để phù hợp với đầu vào của BiLSTM
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # chuẩn hóa layer norm theo từng time-step + tính attention score để tạo vector ngữ cảnh
        out = self.layer_norm(out)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua 2 lớp fully connected với dropout và layer norm để tăng cường ổn định
        out = self.dropout(context_vector)
        out = self.fc1(out) 
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out


# tạo đồ thị GAT cho tập valid và test (cần tách riêng để mô phỏng thực tế (không gộp lại giống lúc train))
def build_inference_graph(df, window_size=10):
    print("Pre-processing Timestamp cho Inference Graph...")
    if 'timestamp_num' not in df.columns:
        df['timestamp_num'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
        
    df.sort_values(by='timestamp_num', inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    cols_present = [c for c in cols_to_drop if c in df.columns]
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
    features_np = np.ascontiguousarray(features_np)
    
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()

    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    ip_to_indices = {}
    from tqdm.auto import tqdm
    for idx in tqdm(range(len(df)), desc="[1/2] Build IP Dictionary"):
        s_ip, d_ip = src_ips[idx], dst_ips[idx]
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx: ip_to_indices[s_ip].append(idx)
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx: ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Nối lưới đồ thị (1-chiều Inference)"):
        n_flows = len(indices)
        if n_flows < 2: continue
        for i in range(1, n_flows):
            curr_idx = indices[i]
            for j in range(max(0, i - window_size), i):
                past_idx = indices[j]
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx])
                if dt_raw > 60.0: continue
                dt = np.log1p(dt_raw * 1e6) / 18.0
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0
                attr = [dt, same_dir, same_sport, same_dport]
                
                all_src.append(past_idx)
                all_dst.append(curr_idx)
                all_edge_attrs.append(attr)
                
    df_edges = pd.DataFrame({'src': all_src, 'dst': all_dst, 'a0': [a[0] for a in all_edge_attrs], 'a1': [a[1] for a in all_edge_attrs], 'a2': [a[2] for a in all_edge_attrs], 'a3': [a[3] for a in all_edge_attrs]})
    df_edges.drop_duplicates(subset=['src', 'dst'], inplace=True)
    df_edges.reset_index(drop=True, inplace=True)
    
    edge_index = torch.tensor(df_edges[['src', 'dst']].values.T, dtype=torch.long).contiguous()
    edge_attr = torch.tensor(df_edges[['a0', 'a1', 'a2', 'a3']].values, dtype=torch.float32).contiguous()
    
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    graph.mask = torch.ones(len(df), dtype=torch.bool) 
    
    del features_np, df_edges, all_src, all_dst, all_edge_attrs
    gc.collect()
    return graph, df

# trích xuất embeddings GAT thực tế bằng NeighborLoader
@torch.no_grad()
def extract_embeddings_mask(model, graph_data, device='cpu'):
    model.eval()
    loader = NeighborLoader(graph_data, num_neighbors=[100, 30], input_nodes=graph_data.mask, batch_size=512, shuffle=False, num_workers=0)
    from tqdm.auto import tqdm
    pbar = tqdm(loader, desc="Extracting GAT Embeddings")


    all_combined = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Nối")
    for batch in pbar:
        batch = batch.to(device)
        
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        bs = batch.batch_size
        
        # lấy raw features và embedding các node gốc
        raw_features = batch.x[:bs].cpu().numpy()
        embeddings = emb[:bs].cpu().numpy()
        
        # concat raw_features và embeddings 
        combined_matrix = np.concatenate([raw_features, embeddings], axis=1)
        
        all_combined.append(combined_matrix)
        
        del batch, emb, _, raw_features, embeddings, combined_matrix
        
    pbar.close()
    return np.concatenate(all_combined, axis=0)

# trích xuất xác suất dự đoán từ mô hình CNN-BiLSTM-Attention
@torch.no_grad()
def extract_probs_cnn_bilstm(model, X_windows, batch_size=512, device='cpu'):
    model.eval()
    import numpy as np

    # Kiểm tra nhanh kích thước -> debug
    try:
        print("X_windows.shape:", X_windows.shape, "dtype:", X_windows.dtype, "bytes:", getattr(X_windows, "nbytes", None))
    except Exception:
        pass

    all_probs = []
    N = len(X_windows)
    for i in range(0, N, batch_size):
        batch_np = X_windows[i:i+batch_size]
        # đảm bảo float32 để giảm 2x bộ nhớ so với float64
        if batch_np.dtype != np.float32:
            batch_np = batch_np.astype(np.float32, copy=False)
        inputs = torch.from_numpy(batch_np).to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

class SlidingWindowDataset(Dataset):
    def __init__(self, df, window_size=10):
        ignore_cols = ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']
        feature_cols = [c for c in df.columns if c not in ignore_cols]
        
        # Chỉ lưu bản gốc 2D trong RAM
        self.features = df[feature_cols].values.astype(np.float32)
        self.labels = df['label'].values
        self.window_size = window_size

    def __len__(self):
        return len(self.features) - self.window_size + 1

    def __getitem__(self, idx):
        # Trích xuất cửa sổ on-the-fly
        window = self.features[idx : idx + self.window_size]
        target_idx = idx + self.window_size - 1
        label = self.labels[target_idx]
        
        return torch.tensor(window), label, target_idx

In [4]:
def extract_probs_cnn_bilstm_v2(model, dataloader, device):
    model.eval()
    all_probs = []
    all_labels = []
    all_indices = []
    
    print("Đang trích xuất xác suất từ CNN-BiLSTM...")
    with torch.no_grad():
        for windows, labels, indices in tqdm(dataloader):
            windows = windows.to(device)
            # Giả sử model trả về logits, dùng softmax để lấy probabilities
            outputs = model(windows) 
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            
            all_probs.append(probs)
            all_labels.extend(labels.numpy())
            all_indices.extend(indices.numpy())
            
    return np.vstack(all_probs), np.array(all_labels), np.array(all_indices)

In [5]:
import numpy as np
from tqdm.auto import tqdm
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# hàm tạo sliding windows với chỉ số mảng để align với output của GAT + XGBoost
'''
def get_window_data_with_index(df, window_size=10):
    """
    trượt cửa sổ bằng chỉ số mảng (numpy array index) đẻ nối output của model (GAT_XGB)
    """
    ignore_cols = ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']
    feature_cols = [c for c in df.columns if c not in ignore_cols]
    
    features = df[feature_cols].values.astype(np.float32)
    labels = df['label'].values
    
    X_windows = []
    y_targets = []
    target_indices = []
    
    print(f"Processing sliding windows (size={window_size})...")
    for i in tqdm(range(len(df) - window_size + 1)):
        window = features[i : i + window_size]
        target_idx = i + window_size - 1
        target_label = labels[target_idx]
        
        X_windows.append(window)
        y_targets.append(target_label)
        target_indices.append(target_idx)
        
    return np.array(X_windows), np.array(y_targets), np.array(target_indices)
'''
# xây dựng đồ thị và trích xuất embeddings GAT cho tập valid và test
device = 'cuda' if torch.cuda.is_available() else 'cpu'

num_features = len([c for c in data_valid.columns if c not in ['label', 'flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']])
num_classes = len(data_valid['label'].unique())

# load gat embedder
print(f"Đang Load trọng số GAT_Embedder (features = {num_features}, classes = {num_classes})...")
gat_model = GAT_Embedder(in_channels=num_features, hidden_channels=128, embedding_dim=64, num_classes=num_classes, heads=8, edge_dim=4).to(device)
gat_model.load_state_dict(torch.load(r"C:\Users\Admin\Downloads\IoT Dataset\tcp_based_training_model\model_final\gat_embedder.pth", map_location=device))

# xây dựng inference graph (tách riêng cho valid và test để mô phỏng thực tế, không gộp lại giống lúc train)
print("\n--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---")
#graph_train, data_train = build_inference_graph(data_train)
graph_valid, data_valid = build_inference_graph(data_valid)
graph_test, data_test = build_inference_graph(data_test)

# trích xuất embeddings từ valid và test
print("\n--- TRÍCH XUẤT EMBEDDINGS TỪ VALID VÀ TEST ---")
#train_gat_emb = extract_embeddings_mask(gat_model, graph_train, device=device)
valid_gat_emb = extract_embeddings_mask(gat_model, graph_valid, device=device)
test_gat_emb  = extract_embeddings_mask(gat_model, graph_test, device=device)

# tạo sliding windows cho CNN-BiLSTM (không có under-sampling, giữ nguyên thứ tự thời gian)
#X_train_win, y_train_win, train_align_idx = get_window_data_with_index(data_train, window_size=10)
#X_val_win, y_val_win, val_align_idx = get_window_data_with_index(data_valid, window_size=10)
#X_test_win, y_test_win, test_align_idx = get_window_data_with_index(data_test, window_size=10)

val_dataset = SlidingWindowDataset(data_valid, window_size=10)
test_dataset = SlidingWindowDataset(data_test, window_size=10)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)


#print(f"\n[Valid] Windows Shape: {X_val_win.shape} | Align Indices: {val_align_idx.shape}")
#print(f"[Test]  Windows Shape: {X_test_win.shape}  | Align Indices: {test_align_idx.shape}")

Đang Load trọng số GAT_Embedder (features = 323, classes = 13)...

--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---
Pre-processing Timestamp cho Inference Graph...


C:\Users\Admin\AppData\Local\Temp\ipykernel_23396\1611056445.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gat_model.load_state_dict(torch.load(r"C:\Users\Admin\Downl

[1/2] Build IP Dictionary:   0%|          | 0/4393262 [00:00<?, ?it/s]

[2/2] Nối lưới đồ thị (1-chiều Inference):   0%|          | 0/21 [00:00<?, ?it/s]

Pre-processing Timestamp cho Inference Graph...


...


--- TRÍCH XUẤT EMBEDDINGS TỪ VALID VÀ TEST ---


Extracting GAT Embeddings:   0%|          | 0/8581 [00:00<?, ?it/s]

Đang trích xuất Vectơ Nối:   0%|          | 0/8581 [00:00<?, ?it/s]

Extracting GAT Embeddings:   0%|          | 0/6436 [00:00<?, ?it/s]

Đang trích xuất Vectơ Nối:   0%|          | 0/6436 [00:00<?, ?it/s]

In [6]:
print("\n--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(r"C:\Users\Admin\Downloads\IoT Dataset\tcp_based_training_model\model_final\GAT_XGB_Hybrid_Concat_Model.json")

#prob_train_xgb = xgb_model.predict_proba(train_gat_emb)
prob_valid_xgb = xgb_model.predict_proba(valid_gat_emb)
prob_test_xgb  = xgb_model.predict_proba(test_gat_emb)

print("\n--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---")
cnn_model = CNN_BiLSTM_Attention(num_features=num_features, num_classes=num_classes, time_steps=10, hidden_size=128).to(device)
cnn_model.load_state_dict(torch.load(r"C:\Users\Admin\Downloads\IoT Dataset\tcp_based_training_model\model_final\best_cnn_bilstm_v2.pth", map_location=device))

# extract probabilities từ CNN-BiLSTM-Attention cho tập valid và test
#prob_train_bilstm = extract_probs_cnn_bilstm(cnn_model, X_train_win, batch_size=512, device=device)
#prob_valid_bilstm = extract_probs_cnn_bilstm(cnn_model, X_val_win, batch_size=512, device=device)
#prob_test_bilstm  = extract_probs_cnn_bilstm(cnn_model, X_test_win, batch_size=512, device=device)
prob_valid_bilstm, y_val_win, val_align_idx = extract_probs_cnn_bilstm_v2(cnn_model, val_loader, device=device)
prob_test_bilstm, y_test_win, test_align_idx = extract_probs_cnn_bilstm_v2(cnn_model, test_loader, device=device)

# rút ra các dự đoán XGBoost (Node level) tương ứng với phần tử cuối cùng của (Sequence Level window)
#aligned_prob_train_xgb = prob_train_xgb[train_align_idx]
aligned_prob_valid_xgb = prob_valid_xgb[val_align_idx]
aligned_prob_test_xgb  = prob_test_xgb[test_align_idx]

# nối đặc trưng meta: XGB Probabilities + CNN-BiLSTM Probabilities

#X_meta_train = np.concatenate([aligned_prob_train_xgb, prob_train_bilstm], axis=1)
#y_meta_train = y_train_win

X_meta_valid = np.concatenate([aligned_prob_valid_xgb, prob_valid_bilstm], axis=1)
y_meta_valid = y_val_win 

X_meta_test = np.concatenate([aligned_prob_test_xgb, prob_test_bilstm], axis=1)
y_meta_test = y_test_win

#print(f"\n[Meta-Dataset] Train Shape:      {X_meta_train.shape}")
print(f"\n[Meta-Dataset] Validation Shape: {X_meta_valid.shape}")
print(f"[Meta-Dataset] Test Shape:       {X_meta_test.shape}")


# meta-learner dùng XGBoost
print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---")

meta_model = xgb.XGBClassifier(
    n_estimators=300,          
    max_depth=3,           
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    early_stopping_rounds=20  
)

meta_model.fit(
    X_meta_valid, y_meta_valid,
    eval_set=[(X_meta_valid, y_meta_valid)],
    verbose=False
)

final_preds = meta_model.predict(X_meta_test)
print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, final_preds, digits=4))


--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---

--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---
Đang trích xuất xác suất từ CNN-BiLSTM...


C:\Users\Admin\AppData\Local\Temp\ipykernel_23396\3403674973.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(r"C:\Users\Admin\Downl

  0%|          | 0/8581 [00:00<?, ?it/s]

Đang trích xuất xác suất từ CNN-BiLSTM...


  0%|          | 0/6436 [00:00<?, ?it/s]


[Meta-Dataset] Validation Shape: (4393253, 26)
[Meta-Dataset] Test Shape:       (3294954, 26)

--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---
Accuracy:         98.93%
F1-Score (Macro): 95.71%

Classification Report:

              precision    recall  f1-score   support

           0     0.9956    0.9999    0.9977    246000
           1     0.9997    1.0000    0.9999    613218
           2     0.9583    0.8743    0.9144     92157
           3     0.9971    1.0000    0.9985    552847
           4     1.0000    0.9271    0.9622    275494
           5     0.9961    1.0000    0.9980    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.8242    0.9600    0.8869     69052
           8     0.9991    1.0000    0.9995    157261
           9     1.0000    0.8298    0.9070      1962
          10     0.9985    0.9896    0.9941      1351
          11     0.6519    0.9844    0.7844     25973
  

In [23]:
# lưu x_meta_train, y_meta_train và x_meta_valid, y_meta_valid, x_meta_test, y_meta_test ra file để huấn luyện meta-learner sau này (nếu cần)
import joblib
#joblib.dump(X_meta_train, "X_meta_train.pkl")
#joblib.dump(y_meta_train, "y_meta_train.pkl")
joblib.dump(X_meta_valid, "X_meta_valid.pkl")
joblib.dump(y_meta_valid, "y_meta_valid.pkl")
joblib.dump(X_meta_test, "X_meta_test.pkl")
joblib.dump(y_meta_test, "y_meta_test.pkl")

['y_meta_test.pkl']

Meta Learner bằng Logistic Regression

In [ ]:
import joblib
X_meta_train = joblib.load("X_meta_train.pkl")
y_meta_train = joblib.load("y_meta_train.pkl")
X_meta_valid = joblib.load("X_meta_valid.pkl")
y_meta_valid = joblib.load("y_meta_valid.pkl")
X_meta_test = joblib.load("X_meta_test.pkl")
y_meta_test = joblib.load("y_meta_test.pkl")

In [ ]:
# huấn luyện meta-learner dùng logistic regression để so sánh với XGBoost meta-learner
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER (LOGISTIC REGRESSION) ---")


best_lr_model = LogisticRegression(
    C=1.0, 
    solver='lbfgs', 
    class_weight='balanced', 
    max_iter=2000, 
    random_state=42
)


print("Đang huấn luyện mô hình Meta-learner trên tập Valid...")
best_lr_model.fit(X_meta_valid, y_meta_valid)

lr_final_preds = best_lr_model.predict(X_meta_test)

print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, lr_final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, lr_final_preds, average='macro')*100:.2f}%")
print("\nClassification Report:\n")   
print(classification_report(y_meta_test, lr_final_preds, digits=4))


--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER (LOGISTIC REGRESSION) ---
Đang huấn luyện mô hình Meta-learner trên tập Valid...

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE ---
Accuracy:         99.40%
F1-Score (Macro): 95.27%

Classification Report:

              precision    recall  f1-score   support

           0     0.9965    0.9997    0.9981    246000
           1     0.9998    0.9984    0.9991    613218
           2     0.9915    0.8655    0.9242     92157
           3     1.0000    1.0000    1.0000    552847
           4     0.9998    0.9990    0.9994    275494
           5     1.0000    1.0000    1.0000    234000
           6     1.0000    1.0000    1.0000    689483
           7     1.0000    0.9240    0.9605     69052
           8     1.0000    1.0000    1.0000    157261
           9     0.8980    0.8344    0.8650      1962
          10     0.8104    1.0000    0.8953      1351
          11     0.5964    0.9889    0.7441     25973
          12     1.0000    1.0000    1.0000    

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_meta_test, tuned_preds)

# Xem những class nào đang bị đoán nhầm thành Class 11
# Cột số 11 trong ma trận là số lượng dự đoán là class 11
predicted_as_11 = cm[:, 11]

print("--- PHÂN TÍCH LỖI CLASS 11 ---")
for actual_class, count in enumerate(predicted_as_11):
    if count > 0:
        print(f"Dữ liệu thật là Class {actual_class:2d} | Bị dự đoán nhầm thành 11: {count} mẫu")

In [8]:
# đánh giá chất lượng nhãn trên tập valid (để xem hiệu quả base learner trước khi stacking)

# 11 cột đầu tiên là xác suất từ XGBoost, 11 cột tiếp theo là xác suất từ CNN-BiLSTM-Attention
x_prob_xgb = X_meta_test[:, :13]  # xác suất từ XGBoost
prob_bilstm = X_meta_test[:, 13:]  # xác suất từ CNN-BiLSTM-Attention
y_true = y_meta_test

print("\n--- ĐÁNH GIÁ CHẤT LƯỢNG NHÃN TRÊN TẬP VALID (TRƯỚC KHI STACKING) ---")
print("\n[Base Learner 1: XGBoost] ---")
print(classification_report(y_true, np.argmax(x_prob_xgb, axis=1), digits=4))
print("\n[Base Learner 2: CNN-BiLSTM-Attention] ---")
print(classification_report(y_true, np.argmax(prob_bilstm, axis=1), digits=4))


--- ĐÁNH GIÁ CHẤT LƯỢNG NHÃN TRÊN TẬP VALID (TRƯỚC KHI STACKING) ---

[Base Learner 1: XGBoost] ---
              precision    recall  f1-score   support

           0     0.9997    0.9932    0.9964    246000
           1     0.9971    0.9979    0.9975    613218
           2     0.9795    0.8934    0.9345     92157
           3     0.9996    1.0000    0.9998    552847
           4     0.9991    0.9997    0.9994    275494
           5     1.0000    1.0000    1.0000    234000
           6     1.0000    1.0000    1.0000    689483
           7     0.9996    0.9360    0.9668     69052
           8     0.9982    1.0000    0.9991    157261
           9     0.9873    0.8344    0.9044      1962
          10     0.8253    1.0000    0.9043      1351
          11     0.6562    0.9834    0.7871     25973
          12     1.0000    1.0000    1.0000    336156

    accuracy                         0.9945   3294954
   macro avg     0.9570    0.9722    0.9607   3294954
weighted avg     0.9958    0.9945